### Train Preprocess Pipeline: Churn from DAC

### 1. Set Kernel Env

In [1]:
# %%bash

# PKGs=$(
#     /opt/conda/envs/python3.9.25_spark_scrach_kernel/bin/pip3 list -v \
#     | grep "/home/$USER/.local/lib/python.*/site-packages" \
#     | awk '{printf "%s==%s ", $1, $2}'
# ) && \
# echo "%pip uninstall -y $PKGs"

In [2]:
# %pip install -e /home/gorelova_i_v/projects/cvm_churn-from-dac_binary-class_churn-dac

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
from pathlib import Path
import sys

sys.path.insert(0, "/home/gorelova_i_v/projects/cvm_churn-from-dac_binary-class_churn-dac/src")

import logging
logger = logging.getLogger()
logger.setLevel(logging.INFO)

from pyspark.sql import SparkSession

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)

import magpie.sql_utils as su

from cvm_model.io import State
import cvm_model.utils as utils
import cvm_model.sql as sql
from cvm_model.parameters import (
    features_for_outliers,
    aud_table,
    fav_omni_features_table,
    preperiod_months,
    features,
    input_suffix,
    template,
)

### 2. State and Connections

In [ ]:
state = State.from_env()
engine = state.credentials.loyalty_gp.sa_engine
s3 = su.get_s3_client()

engine

### 3. Set Event Timestamp

In [5]:
event_timestamp = datetime(2026, 1, 1)

In [6]:
base_month = event_timestamp.date().replace(day=1).isoformat()
target_month = (pd.Timestamp(base_month) + pd.DateOffset(months=1)).date().isoformat()
feature_date = target_month

base_month, target_month, feature_date

('2026-01-01', '2026-02-01', '2026-02-01')

### 4. Resolve S3 Input Path

In [8]:
# input_prefix = state.settings.preprocess_prefix(event_timestamp) / input_suffix
# input_bucket = input_prefix.split('//')[1].split('/')[0]
# input_key = '/'.join(input_prefix.split('//')[1].split('/')[1:])
# input_path = template.format(bucket=input_bucket, prefix=input_key)

# input_bucket, input_key, input_path

### 5. Audience Table

Аудиторией являются пользователи, которые имеют статус DAC в `base_month`, определяемый
по `event_timestamp`.


При этом статус DAC определяется актуальной бизнес-логикой:

```SQL
   has_transaction_activity = 1
   and (
       has_mobapp_activity = 1
       or has_pwa_activity = 1
       or vcoff_trn_cnt > 0
       )
```

Для EDA и базового анализа фичей принимается случайная подвыборка из суммарной аудитории DAC в `base_month`.

In [ ]:
# Optional: create sample audience table once, then keep reusing it.
# Do not rerun random sampling during feature loading, otherwise df and aud_query will diverge.

n_sample = 100_000

sample_aud_query = sql.aud_query + f"""
order by random()
limit {n_sample}
"""

print(sample_aud_query.format(base_month=base_month)[:1000])

In [ ]:
aud_query = f"""
select distinct contact_id :: int as contact_id
from {aud_table}
"""

df = su.execute_custom_query_gp(aud_query)
df = df.astype({col: np.int32 for col in {'contact_id'} & set(df.columns)})

print(df.shape)
print(df['contact_id'].nunique())
assert df['contact_id'].is_unique

df.head()

In [ ]:
# Audience is expected to be already saved in aud_table.
# If you need to recreate it, run sample_aud_query once and save the result to aud_table before this notebook step.

In [ ]:
aud_query

### 6. Target Feature

`target_churn_from_dac = 1` - клиент был DAC в базовом месяце и не DAC в следующем.

In [ ]:
df_part = su.execute_custom_query_gp(
    sql.target_query.format(
        aud=aud_query,
        target_month=target_month,
    )
).fillna(0)

cast_cols = {'is_dac_next_month', 'target_churn_from_dac'} & set(df_part.columns)
df_part = df_part.astype({col: np.int32 for col in cast_cols})

print(df_part.shape)
print(df_part['contact_id'].nunique())
assert df_part['contact_id'].is_unique

df = df.merge(df_part, on='contact_id', how='inner')
print('After target:', df.shape)
display(df['target_churn_from_dac'].value_counts(dropna=False).to_frame('cnt'))
display(df['target_churn_from_dac'].value_counts(normalize=True, dropna=False).to_frame('share'))

df.head()

## Load all features

Recency and all feature blocks are loaded in one cell. Uplift is excluded.


In [ ]:
query_kwargs = dict(
    aud=aud_query,
    date=feature_date,
    month=preperiod_months[0],
)

df_part = su.execute_custom_query_gp(sql.recency_query.format(**query_kwargs))

print(df_part.shape)
print(df_part['contact_id'].nunique())
assert df_part['contact_id'].is_unique

df = df.merge(df_part, on='contact_id', how='left')
print('After recency:', df.shape)

df = utils.load_features(
    engine=engine,
    df=df,
    aud_query=aud_query,
    date=feature_date,
    fav_omni_features_table=fav_omni_features_table,
    preperiod_months=preperiod_months,
    sql_module=sql,
)

print('After all features:', df.shape)
missing_features = set(features) - set(df.columns)
assert len(missing_features) == 0, f'В датасете не хватает фичей: {missing_features}'

df.head()

## Quick checks


In [ ]:
print('Rows:', len(df))
print('Unique contact_id:', df['contact_id'].nunique())
assert df['contact_id'].is_unique

display(df[features].isna().mean().sort_values(ascending=False).head(30).to_frame('na_share'))
display(df['target_churn_from_dac'].value_counts(normalize=True, dropna=False).to_frame('share'))